# Case Study: Analyse von Kundendaten mit Pandas

In [ ]:
# Importieren der Bibliothek

import pandas as pd

In [ ]:
# Einlesen des Datensatzes

df = pd.read_csv('customers.csv') # falls im gleichen Ordner, ansonsten Dateipfad angeben

In [ ]:
df.head(10) # Zahl innerhalb der Klammer: Anzahl der angezeigten Zeilen

In [ ]:
# Größe des Datensatzes

df.shape

In [ ]:
# Spalten des Datensatzes

df.columns

In [ ]:
# Aufrufen einzelner Spalten

df['Churn'] # doppelte Klammer für Dataframe

In [ ]:
# Andern des Datentyps

df['Churn'] = pd.Categorical(df['Churn'])

In [ ]:
# Überblick über die einzelnen Kategorien

df['gender'].unique(), df['Contract'].unique(), df['PaymentMethod'].unique(), df['PhoneService'].unique(), df['InternetService'].unique()

In [ ]:
# Generelle Infos über den Datensatz: Spalten, fehlende Werte, Datentypen

df.info()

In [ ]:
# Anzeigen der Zeilen mit den fehlenden Werten, NaN: Not a Number

df_null = df[df.isnull().any(axis=1)]

df_null

In [ ]:
# Option 1: Entfernen der Zeilen

df.dropna(inplace=True)

In [ ]:
df.info()

In [ ]:
# Option 2: MonthlyCharges = TotalCharges / Option 3: Auffüllen mit 0

df = pd.read_csv('customers.csv') # Erneutes Einlesen des Datensatzes
df['TotalCharges'] = df['TotalCharges'].fillna(df['MonthlyCharges'])
#df['TotalCharges'] = df['TotalCharges'].fillna(0)
nan_indices = df_null.index
df.iloc[nan_indices]

In [ ]:
df.info()

In [ ]:
# Statistik über numerische Variablen

df.describe()

In [ ]:
# Gesamteinnahmen aller Kunden

df['TotalCharges'].sum() # Aufsummieren über eine Spalte

### Überblick einzelne Kunden

In [ ]:
# Kunde mit höchsten gesamten Umsatz

df.loc[df['TotalCharges'].idxmax()]

In [ ]:
# Kundin mit höchsten monatlichen Kosten

df.loc[df['MonthlyCharges'].idxmax()]

In [ ]:
# Kunde mit längster Laufzeit

df.loc[df['tenure'].idxmax()]

In [ ]:
# Kunde mit höchsten gesamten Umsatz im Datensatz

df[df.TotalCharges == df.TotalCharges.max()]

In [ ]:
# Kundin mit höchsten monatlichen Umsatz

df[df.MonthlyCharges == df.MonthlyCharges.max()]

In [ ]:
# Kunden mit längster Laufzeit (im Gegensatz zum idxmax-Befehl: Anzeigen aller Kunden)

df[df.tenure == df.tenure.max()]

### Überblick einzelne Kategorien

In [ ]:
# Anzahl von Telefonverträgen

df['PhoneService'].value_counts()

In [ ]:
# Anzahl von Internetverträgen (DSL/Glasfaser)

df['InternetService'].value_counts()

In [ ]:
# Art der Verträge 

df['Contract'].value_counts()

In [ ]:
# Bezahlmethode

df['PaymentMethod'].value_counts()

In [ ]:
# Kunde gekündigt?

df['Churn'].value_counts()

### Analyse von Churn

In [ ]:
# Wie unterscheidet sich die durchschnittliche Vertragslaufzeit und Zahlungen von Churn/Non-Churn?

df.groupby('Churn').tenure.mean()

In [ ]:
df.groupby('Churn').MonthlyCharges.mean()

In [ ]:
df.groupby('Churn').TotalCharges.mean()

In [ ]:
# Aggregierte Form

df.groupby('Churn').agg({
    'tenure': ['median'],  # oder z.B. median
    'MonthlyCharges': ['mean'],
    'TotalCharges': ['mean']
})

In [ ]:
# Gibt es Unterschiede abhängig vom Geschlecht?

df.groupby(['Churn', 'gender']).size() # Gruppengröße

In [ ]:
# Hat der Telefonvertrag Einfluss auf Churn?

df.groupby(['Churn', 'PhoneService']).size()

In [ ]:
# Hat der Internetservice Einfluss auf Churn?

df.groupby(['Churn', 'InternetService']).size()

In [ ]:
# Übersichtlichere Darstellung

churn_internet = df.groupby(['Churn', 'InternetService']).size().unstack()
churn_internet

In [ ]:
# Prozentualer Anteil

perc = churn_internet.div(churn_internet.sum(axis=1), axis=0) * 100 # Summe über axis=1 (Columns), Division über axis=0 (Zeilen)
perc.round(2) # auf 2 Nachkommastellen runden

In [ ]:
# Hat die Vertragsart Einfluss auf Churn?

churn_contract = df.groupby(['Churn', 'Contract']).size().unstack()
perc = churn_contract.div(churn_contract.sum(axis=1), axis=0) * 100 
perc.round(2)

In [ ]:
# Hat die Bezahlmethode Einfluss auf Churn?

churn_payment = df.groupby(['Churn', 'PaymentMethod']).size().unstack()
perc = churn_payment.div(churn_payment.sum(axis=1), axis=0) * 100 
perc.round(2)

In [ ]:
# Filtern nach Electronic Check, Unterschiede monatliche Zahlung? 

df[df['PaymentMethod'] == 'Electronic check'].groupby('Churn')['MonthlyCharges'].mean()

### Fokus auf Kundengruppen

In [ ]:
# Erstellen eines neuen Datensatzes mit den angewandten Filtern

df_2 = df[(df['Churn'] == 'Yes') & (df['PaymentMethod'] == 'Electronic check') & (df['InternetService'] == 'Fiber optic')]
df_2.head(20)

In [ ]:
# Entfernen der Spalten Churn, PaymentMethod und InternetService, da diese feststehen

df_2 = df_2.drop(['Churn', 'PaymentMethod', 'InternetService'], axis=1)
df_2.head()

In [ ]:
df_2.shape

In [ ]:
# Vergleich zum ursprünglichen Datensatz

df.describe()

In [ ]:
df_2.describe()

In [ ]:
# Erstellen von Kundenkategorien basierend auf TotalCharges

df_a = df_2[df_2['TotalCharges'] < 220.6]
df_b = df_2[(df_2['TotalCharges'] >= 220.6) & (df_2['TotalCharges'] < 952.3)]
df_c = df_2[(df_2['TotalCharges'] >= 952.3) & (df_2['TotalCharges'] < 2510.2)]
df_d = df_2[df_2['TotalCharges'] >= 2510.2]

In [ ]:
# Größe der einzelnen Cluster

df_a.shape, df_b.shape, df_c.shape, df_d.shape

In [ ]:
df_a.sort_values(by='TotalCharges', ascending=True)

In [ ]:
# Anwendung einer Hilfsfunktion auf den Datensatz

def assign_group(TotalCharges):
    if TotalCharges < 220.6:
        return 'Group A'
    elif 220.6 <= TotalCharges < 952.3:
        return 'Group B'
    elif 952.3 <= TotalCharges < 2510.2:
        return 'Group C'
    else:
        return 'Group D'

df_2['CustomerGroup'] = df_2['TotalCharges'].apply(assign_group)
df_2.head()

In [ ]:
# Anzeigen der Gruppengröße

group_counts = df_2.groupby('CustomerGroup').size().sort_values(ascending=False)
group_counts

In [ ]:
# Hinzufügen der Gruppengröße Information an den Datensatz (Feature Engineering)

group_counts_df = group_counts.reset_index() # group counts als Dataframe
group_counts_df.columns = ['CustomerGroup', 'group_counts']

df_2 = pd.merge(df_2, group_counts_df, on='CustomerGroup', how='left') # Merge

df_2.head()

In [ ]:
# Ändern der Namen von Kategorien

df_2['CustomerGroup'] = df_2['CustomerGroup'].map({'Group A': 'A', 'Group B': 'B', 'Group C': 'C', 'Group D': 'D'})
df_2.head()

In [ ]:
# Entfernen von einzelnen Kategorien (z.B. einzelne Kundengruppen)

df_2_c_d = df_2[df_2['CustomerGroup'].isin(['C', 'D'])]
df_2_c_d.head()

In [ ]:
# Weitere Möglichkeit (z.B. bei sehr vielen Kategorien)

df_2_c_d_2 = df_2[~df_2['CustomerGroup'].isin(['A', 'B'])]
df_2_c_d_2.head()